In [ ]:
#!/usr/bin/env python3
"""
repack_and_upload.py
--------------------
Downloads ShreyashDhoot/KTO-latents from Hugging Face, repacks each split
into Parquet files capped at 200 MB, deletes the old parquet files from the
repo, and uploads the new ones — preserving the split subdirectory layout.

Usage:
    HF_TOKEN=hf_... python repack_and_upload.py

Requirements:
    pip install huggingface-hub pyarrow tqdm
"""

from __future__ import annotations

import os
import math
import shutil
import tempfile
import glob
from pathlib import Path

import pyarrow as pa
import pyarrow.parquet as pq
from huggingface_hub import HfApi, snapshot_download
from tqdm import tqdm

# ─── Configuration ────────────────────────────────────────────────────────────
REPO_ID        = "ShreyashDhoot/KTO-latents"
REPO_TYPE      = "dataset"
SPLITS         = ["train", "val"]          # subdirectories inside the repo
MAX_FILE_BYTES = 200 * 1024 * 1024         # 200 MB hard cap per output file
CACHE_DIR      = "./hf_cache_repack"       # local download cache
HF_TOKEN       = os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN")
# ──────────────────────────────────────────────────────────────────────────────


def download_repo(cache_dir: str) -> str:
    """Download the full dataset repo and return the local root path."""
    print(f"[1/4] Downloading {REPO_ID} …")
    local_dir = snapshot_download(
        repo_id=REPO_ID,
        repo_type=REPO_TYPE,
        local_dir=cache_dir,
        local_dir_use_symlinks=False,
        token=HF_TOKEN,
    )
    print(f"      Saved to: {local_dir}")
    return local_dir


def estimate_row_bytes(table: pa.Table, sample_rows: int = 200) -> float:
    """Estimate average bytes per row by serialising a small sample."""
    n = min(sample_rows, len(table))
    if n == 0:
        return 1.0
    sample = table.slice(0, n)
    buf = pa.BufferOutputStream()
    pq.write_table(sample, buf, compression="snappy")
    return buf.getvalue().size / n


def repack_split(local_root: str, split: str, out_dir: Path) -> list[Path]:
    """
    Read all parquet files for a split, repack them into chunks ≤ MAX_FILE_BYTES,
    write to out_dir, and return the list of output paths.
    """
    split_dir = Path(local_root) / split
    if not split_dir.exists():
        # Some repos keep files directly at root without subdirs
        split_dir = Path(local_root)

    src_files = sorted(split_dir.glob("*.parquet"))
    if not src_files:
        print(f"  [!] No parquet files found for split '{split}' in {split_dir}")
        return []

    print(f"\n[2/4] Repacking split '{split}' — {len(src_files)} source file(s) …")

    # Read all source files into a single concatenated table
    tables = []
    for f in tqdm(src_files, desc=f"  reading {split}", unit="file"):
        tables.append(pq.read_table(f))
    full_table = pa.concat_tables(tables)
    total_rows = len(full_table)
    print(f"      Total rows: {total_rows:,}")

    # Estimate bytes per row from a small sample, then compute chunk size
    bytes_per_row = estimate_row_bytes(full_table)
    rows_per_chunk = max(1, math.floor(MAX_FILE_BYTES / bytes_per_row))
    n_chunks = math.ceil(total_rows / rows_per_chunk)
    print(f"      ~{bytes_per_row:.0f} B/row → {rows_per_chunk:,} rows/chunk → {n_chunks} file(s)")

    out_dir.mkdir(parents=True, exist_ok=True)
    output_paths: list[Path] = []

    for chunk_idx in tqdm(range(n_chunks), desc=f"  writing {split}", unit="file"):
        start = chunk_idx * rows_per_chunk
        end   = min(start + rows_per_chunk, total_rows)
        chunk = full_table.slice(start, end - start)

        out_path = out_dir / f"data_{chunk_idx:04d}.parquet"
        pq.write_table(chunk, out_path, compression="snappy")

        actual_mb = out_path.stat().st_size / 1024 / 1024
        # If the estimate was off and a chunk is still too large, split further
        if out_path.stat().st_size > MAX_FILE_BYTES:
            print(f"      [!] chunk {chunk_idx} is {actual_mb:.1f} MB — splitting further …")
            out_path.unlink()
            half = len(chunk) // 2
            for sub_idx, sub_slice in enumerate([chunk.slice(0, half), chunk.slice(half)]):
                sub_path = out_dir / f"data_{chunk_idx:04d}_{sub_idx}.parquet"
                pq.write_table(sub_slice, sub_path, compression="snappy")
                output_paths.append(sub_path)
        else:
            output_paths.append(out_path)

    total_out_mb = sum(p.stat().st_size for p in output_paths) / 1024 / 1024
    print(f"      Written {len(output_paths)} file(s), {total_out_mb:.1f} MB total")
    return output_paths


def delete_old_parquets(api: HfApi, split: str) -> None:
    """Delete all existing .parquet files under the split folder on the Hub."""
    print(f"\n[3/4] Deleting old parquet files for split '{split}' from Hub …")
    try:
        repo_files = api.list_repo_files(
            repo_id=REPO_ID,
            repo_type=REPO_TYPE,
            token=HF_TOKEN,
        )
    except Exception as e:
        print(f"  [!] Could not list repo files: {e}")
        return

    to_delete = [
        f for f in repo_files
        if f.startswith(f"{split}/") and f.endswith(".parquet")
    ]

    if not to_delete:
        print(f"  No existing parquet files found under '{split}/' — skipping deletion.")
        return

    print(f"  Deleting {len(to_delete)} file(s) …")
    operations = [
        {"path_in_repo": path, "path_or_fileobj": None}
        for path in to_delete
    ]

    # HfApi.delete_files uses CommitOperationDelete
    from huggingface_hub import CommitOperationDelete
    delete_ops = [CommitOperationDelete(path_in_repo=p) for p in to_delete]
    api.create_commit(
        repo_id=REPO_ID,
        repo_type=REPO_TYPE,
        operations=delete_ops,
        commit_message=f"repack: delete old {split}/ parquet files",
        token=HF_TOKEN,
    )
    print(f"  Deleted {len(to_delete)} file(s).")


def upload_new_parquets(api: HfApi, split: str, output_paths: list[Path]) -> None:
    """Upload repacked parquet files to the Hub under split/."""
    if not output_paths:
        return
    print(f"\n[4/4] Uploading {len(output_paths)} repacked file(s) for split '{split}' …")

    from huggingface_hub import CommitOperationAdd

    add_ops = []
    for local_path in tqdm(output_paths, desc=f"  preparing {split}", unit="file"):
        repo_path = f"{split}/{local_path.name}"
        add_ops.append(
            CommitOperationAdd(
                path_in_repo=repo_path,
                path_or_fileobj=str(local_path),
            )
        )

    api.create_commit(
        repo_id=REPO_ID,
        repo_type=REPO_TYPE,
        operations=add_ops,
        commit_message=f"repack: upload repacked {split}/ parquet files (max 200 MB each)",
        token=HF_TOKEN,
    )
    print(f"  Uploaded {len(output_paths)} file(s) to {REPO_ID}/{split}/")


def main() -> None:
    if not HF_TOKEN:
        raise EnvironmentError(
            "Set HF_TOKEN (or HUGGING_FACE_HUB_TOKEN) to a token with write access to the repo."
        )

    api = HfApi()

    # Step 1: Download
    local_root = download_repo(CACHE_DIR)

    with tempfile.TemporaryDirectory(prefix="repack_out_") as tmp_out:
        tmp_out_path = Path(tmp_out)

        for split in SPLITS:
            # Step 2: Repack
            split_out_dir = tmp_out_path / split
            output_paths = repack_split(local_root, split, split_out_dir)

            if not output_paths:
                print(f"  Skipping upload for split '{split}' — no output files.")
                continue

            # Step 3: Delete old files on Hub
            delete_old_parquets(api, split)

            # Step 4: Upload new files
            upload_new_parquets(api, split, output_paths)

    print("\nDone! All splits repacked and re-uploaded.")
    print(f"Repo: https://huggingface.co/datasets/{REPO_ID}")


if __name__ == "__main__":
    main()

[1/4] Downloading ShreyashDhoot/KTO-latents …


d:\anaconda\envs\maskdraw\lib\site-packages\huggingface_hub\utils\_validators.py:202: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `snapshot_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


Fetching 7 files:   0%|          | 0/7 [00:00<?, ?it/s]

In [2]:
!pip install huggingface.hub

  Using cached hf_xet-1.4.3-cp37-abi3-win_amd64.whl.metadata (4.9 kB)


ERROR: Could not install packages due to an OSError: [Errno 2] No such file or directory: 'd:\\anaconda\\lib\\site-packages\\httpcore-1.0.2.dist-info\\METADATA'

